<a href="https://colab.research.google.com/github/attilagk/VPS13A-deep-learning/blob/main/notebooks/2025-11-13-DL-hello-world.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#%load_ext autoreload
#%autoreload 2
#%reload_ext autoreload
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision.transforms import ToTensor
from torchvision import datasets

In [2]:
torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0)

('2.12.0.dev20260304+cu128', True, 'NVIDIA GeForce RTX 5060 Ti')

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [4]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))
    print("arch list:", torch.cuda.get_arch_list())
    x = torch.randn(4096, 4096, device="cuda")
    y = torch.randn(4096, 4096, device="cuda")
    z = x @ y
    torch.cuda.synchronize()
    print("matmul ok:", z.shape)

torch: 2.12.0.dev20260304+cu128
cuda available: True
gpu: NVIDIA GeForce RTX 5060 Ti
capability: (12, 0)
arch list: ['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
matmul ok: torch.Size([4096, 4096])


# Hello World!

From the PyTorch tutorial: [Optimizing Model Parameters](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)

In [5]:
training_data = datasets.FashionMNIST(
    root='../data',
    train=True,
    download=True,
    transform=ToTensor(),
)

In [6]:
test_data = datasets.FashionMNIST(
    root='../data',
    train=False,
    download=True,
    transform=ToTensor(),
)

In [7]:
loader_kwargs = {"batch_size": 64}
if device == "cuda":
    loader_kwargs.update({"pin_memory": True, "num_workers": 4})

train_dataloader = DataLoader(training_data, shuffle=True, **loader_kwargs)
test_dataloader = DataLoader(test_data, batch_size=64)

In [8]:
class HelloWorldNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = HelloWorldNN().to(device)

In [9]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [10]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        # Compute loss
        pred = model(X)
        loss = loss_fn(pred, y)
        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        # Print progress
        if batch % 100 == 0:
            loss = loss.item()
            current = batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

        test_loss /= num_batches
        correct /= size
        print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [11]:
epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)

Epoch 1
-------------------------------


loss: 2.312412  [   64/60000]
loss: 2.288406  [ 6464/60000]


loss: 2.270490  [12864/60000]
loss: 2.263843  [19264/60000]


loss: 2.243048  [25664/60000]
loss: 2.236335  [32064/60000]


loss: 2.218356  [38464/60000]
loss: 2.213987  [44864/60000]
loss: 2.185423  [51264/60000]


...

loss: 0.809533  [   64/60000]
loss: 0.788218  [ 6464/60000]
loss: 0.827986  [12864/60000]


loss: 0.812842  [19264/60000]
loss: 0.793572  [25664/60000]


loss: 0.793560  [32064/60000]
loss: 0.765971  [38464/60000]
loss: 0.842250  [44864/60000]


loss: 0.637680  [51264/60000]
loss: 0.688534  [57664/60000]


Test Error: 
 Accuracy: 71.4%, Avg loss: 0.785401 



In [12]:
%connect_info

{
  "shell_port": 33873,
  "iopub_port": 41191,
  "stdin_port": 44941,
  "control_port": 50121,
  "hb_port": 37871,
  "ip": "127.0.0.1",
  "key": "34dbe94b-e7f5205f63c6c3bb57ef80d5",
  "transport": "tcp",
  "signature_scheme": "hmac-sha256",
  "kernel_name": "dl-cuda-nightly"
}

Paste the above JSON into a file, and connect with:
    $> jupyter <app> --existing <file>
or, if you are local, you can connect with just:
    $> jupyter <app> --existing /tmp/tmpfauo6e1k.json
or even just:
    $> jupyter <app> --existing
if this is the most recent Jupyter kernel you have started.
